# Sales Anomaly Analysis — EDA & Preprocessing

**데이터:** 글로벌 완구·모형 판매 데이터 (2003–2005, 2,823건)  
**목적:** 이상치 탐지를 위한 탐색적 분석 및 피처 엔지니어링

## 0. 환경 설정

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['figure.dpi'] = 100

## 1. 데이터 로딩

In [ ]:
file_path = '../data/sales_data_sample.csv'
df = pd.read_csv(file_path, encoding='unicode_escape')
print(f"데이터 로드 성공! shape: {df.shape}")
display(df.head())

## 2. 데이터 탐색

### 2.1 기본 구조 확인

In [ ]:
print("=== 컬럼 목록 ===")
print(df.columns.tolist())
print()
df.info()

In [ ]:
display(df.describe())

### 2.2 범주형 변수 분포

In [ ]:
print("--- 주요 범주별 고유값 개수 ---")
cols_to_check = ['COUNTRY', 'PRODUCTLINE', 'STATUS', 'DEALSIZE', 'CUSTOMERNAME']
for col in cols_to_check:
    print(f"{col}: {df[col].nunique()}개")

print("\n--- 국가별 거래 건수 Top 10 ---")
print(df['COUNTRY'].value_counts().head(10))

print("\n--- 제품군별 거래 건수 ---")
print(df['PRODUCTLINE'].value_counts())

### 2.3 날짜 범위 및 중복 확인

In [ ]:
temp_dates = pd.to_datetime(df['ORDERDATE'])
print(f"시작일: {temp_dates.min().date()}")
print(f"종료일: {temp_dates.max().date()}")
print(f"중복 행: {df.duplicated().sum()}개")

### 2.4 수치형 변수 간 상관관계 (SALES 기준)

In [ ]:
numeric_df = df.select_dtypes(include=['int64', 'float64'])
print("--- SALES 기준 상관계수 ---")
print(numeric_df.corr()['SALES'].sort_values(ascending=False))

## 3. 데이터 품질 확인 — 결측치

In [ ]:
null_info = pd.DataFrame({
    'Null Count': df.isnull().sum(),
    'Null Percentage (%)': (df.isnull().sum() / len(df)) * 100
})
null_summary = null_info[null_info['Null Count'] > 0].sort_values('Null Count', ascending=False)
print("--- 컬럼별 결측치 현황 ---")
display(null_summary)

# 시각화
null_data = (df.isnull().sum() / len(df) * 100)
null_data = null_data[null_data > 0].sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=null_data.index, y=null_data.values, color='#378ADD')
plt.title('Missing Value Rate by Column (%)', fontsize=14, fontweight='bold')
plt.ylabel('Missing Rate (%)')
plt.xticks(rotation=30)
for i, v in enumerate(null_data.values):
    plt.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('../figures/fig02_null_check.png', bbox_inches='tight')
plt.show()

## 4. 데이터 전처리

In [ ]:
# 날짜 변환
df['ORDERDATE'] = pd.to_datetime(df['ORDERDATE'])

# 분석에 불필요한 컬럼 삭제
#  - ADDRESSLINE2: 결측 89.3%
#  - PHONE, CONTACTLASTNAME, CONTACTFIRSTNAME: 개인정보성
#  - POSTALCODE, STATE: 분석 미사용
drop_cols = ['ADDRESSLINE2', 'PHONE', 'CONTACTLASTNAME', 'CONTACTFIRSTNAME', 'POSTALCODE', 'STATE']
df_cleaned = df.drop(columns=drop_cols)

print(f"전처리 전 컬럼 수: {df.shape[1]}")
print(f"전처리 후 컬럼 수: {df_cleaned.shape[1]}")
print(f"유지된 컬럼: {df_cleaned.columns.tolist()}")

## 5. EDA (탐색적 데이터 분석)

### 5.1 수치형 변수 분포

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Distribution of Key Numerical Variables', fontsize=16, fontweight='bold')

for ax, col, color in zip(axes.flatten(),
                          ['SALES', 'QUANTITYORDERED', 'PRICEEACH', 'MSRP'],
                          ['#378ADD', '#E24B4A', '#1D9E75', '#BA7517']):
    sns.histplot(df_cleaned[col], kde=True, ax=ax, color=color)
    ax.set_title(col)

plt.tight_layout()
plt.savefig('../figures/fig01_distribution.png', bbox_inches='tight')
plt.show()

### 5.2 상관관계 히트맵

In [ ]:
numeric_cols = df_cleaned.select_dtypes(include=['float64', 'int64']).columns
corr_matrix = df_cleaned[numeric_cols].corr()

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f',
            linewidths=0.5, ax=axes[0])
axes[0].set_title('Correlation Heatmap of Numerical Features', fontsize=14, fontweight='bold')

sales_corr = corr_matrix['SALES'].drop('SALES').sort_values(ascending=False)
sns.barplot(x=sales_corr.values, y=sales_corr.index, ax=axes[1], color='#378ADD')
axes[1].set_title('Impact on SALES by Feature', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Correlation Coefficient')
for i, v in enumerate(sales_corr.values):
    axes[1].text(v + 0.01, i, f'{v:.2f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('../figures/fig06_correlation_heatmap.png', bbox_inches='tight')
plt.show()

### 5.3 연도별 매출 추이

In [ ]:
plt.figure(figsize=(9, 5))
df_cleaned.groupby('YEAR_ID')['SALES'].sum().plot(kind='bar', color='#378ADD', edgecolor='white')
plt.title('Total Sales by Year', fontsize=14, fontweight='bold')
plt.ylabel('Total Sales ($)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../figures/fig04_sales_by_year.png', bbox_inches='tight')
plt.show()
print("※ 2005년은 5월까지 데이터만 존재 — 연도 간 직접 비교 불가")

### 5.4 제품군별 매출 분포

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(x='PRODUCTLINE', y='SALES', data=df_cleaned, color='#378ADD')
plt.title('Sales Distribution by Product Line', fontsize=14, fontweight='bold')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('../figures/fig05_sales_by_productline.png', bbox_inches='tight')
plt.show()

### 5.5 국가별 매출 비중

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

country_sales = df_cleaned.groupby('COUNTRY')['SALES'].sum().sort_values()
country_sales.plot(kind='barh', ax=axes[0], color='#1D9E75')
axes[0].set_title('Total Sales by Country', fontsize=13, fontweight='bold')

top5 = df_cleaned.groupby('COUNTRY')['SALES'].sum().nlargest(5).index
df_top5 = df_cleaned[df_cleaned['COUNTRY'].isin(top5)]
monthly = df_top5.groupby(['COUNTRY', 'YEAR_ID', 'MONTH_ID'])['SALES'].mean().reset_index()
monthly['PERIOD'] = monthly['YEAR_ID'].astype(str) + '-' + monthly['MONTH_ID'].astype(str).str.zfill(2)
sns.lineplot(data=monthly.sort_values('PERIOD'), x='PERIOD', y='SALES',
             hue='COUNTRY', marker='o', ax=axes[1])
axes[1].set_title('Monthly Sales Trends (Top 5 Countries)', fontsize=13, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.savefig('../figures/fig03_sales_by_country.png', bbox_inches='tight')
plt.show()

## 6. 다중공선성 검증 (VIF)

파생변수 설계 전 원본 변수 간 중복 정보 확인

In [ ]:
vif_cols = ['SALES', 'QUANTITYORDERED', 'PRICEEACH', 'MSRP']
vif_data = df_cleaned[vif_cols].dropna()
vif_df = pd.DataFrame({
    'Feature': vif_data.columns,
    'VIF': [variance_inflation_factor(vif_data.values, i) for i in range(len(vif_data.columns))]
})
print("--- Variance Inflation Factor (VIF) Analysis ---")
display(vif_df.sort_values('VIF', ascending=False))

## 7. 이상치 탐지 — 국가별 IQR

**방법론 선택 근거:** 글로벌 데이터에 단일 IQR 적용 시 국가별 매출 편차로 인한 오분류 발생.  
국가별로 독립적인 Q1, Q3, IQR을 산출하여 `Q3 + 1.5 × IQR` 기준 적용.

### 7.1 국가별 매출 분포 및 이상치 탐지

In [ ]:
plt.figure(figsize=(16, 7))
sns.boxplot(data=df_cleaned, x='COUNTRY', y='SALES',
            hue='COUNTRY', palette='Set3', legend=False)
plt.title('Outlier Detection by Country (Boxplot & IQR)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../figures/fig07_boxplot_by_country.png', bbox_inches='tight')
plt.show()

def find_outliers(group):
    q1 = group['SALES'].quantile(0.25)
    q3 = group['SALES'].quantile(0.75)
    iqr = q3 - q1
    return group[(group['SALES'] > q3 + 1.5 * iqr) | (group['SALES'] < q1 - 1.5 * iqr)]

all_outliers = df_cleaned.groupby('COUNTRY', group_keys=False).apply(find_outliers)
print(f"탐지된 총 이상치: {len(all_outliers)}건 ({len(all_outliers)/len(df_cleaned)*100:.2f}%)")

### 7.2 리스크 거래 식별 (Disputed / Cancelled)

In [ ]:
if 'COUNTRY' not in all_outliers.columns:
    outliers_fixed = all_outliers.reset_index()
else:
    outliers_fixed = all_outliers.copy()

bad_status = outliers_fixed[outliers_fixed['STATUS'].isin(['Disputed', 'Cancelled'])]
print(f"--- Disputed / Cancelled 이상치: {len(bad_status)}건 ---")
display(bad_status[['COUNTRY', 'ORDERNUMBER', 'CUSTOMERNAME', 'SALES', 'STATUS', 'ORDERDATE']])

status_counts = all_outliers['STATUS'].value_counts()
plt.figure(figsize=(10, 5))
palette_map = {'Shipped':'#378ADD','Disputed':'#E24B4A','Cancelled':'#A32D2D',
               'On Hold':'#BA7517','In Process':'#1D9E75','Resolved':'#637074'}
sns.barplot(x=status_counts.index, y=status_counts.values,
            palette=[palette_map.get(x, '#888') for x in status_counts.index])
plt.title('Status Distribution within Outliers', fontsize=14, fontweight='bold')
plt.ylabel('Count')
for i, v in enumerate(status_counts.values):
    plt.text(i, v + 0.3, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/fig08_outlier_status.png', bbox_inches='tight')
plt.show()

### 7.3 제품군별 이상치 집중도

In [ ]:
outlier_product = all_outliers['PRODUCTLINE'].value_counts()
outlier_pct = all_outliers['PRODUCTLINE'].value_counts(normalize=True) * 100
summary_df = pd.DataFrame({'Count': outlier_product, 'Percentage (%)': outlier_pct})
print("--- Outlier Distribution by Product Line ---")
display(summary_df)

plt.figure(figsize=(11, 5))
sns.countplot(data=all_outliers, x='PRODUCTLINE', order=outlier_product.index, color='#378ADD')
plt.title('Number of Outliers by Product Line', fontsize=14, fontweight='bold')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('../figures/fig09_outlier_by_productline.png', bbox_inches='tight')
plt.show()

### 7.4 고객 충성도 분석 — 이상치 고객은 VIP인가?

In [ ]:
outlier_customers = all_outliers['CUSTOMERNAME'].unique()

customer_behavior = df_cleaned.groupby('CUSTOMERNAME').agg(
    Total_Orders=('ORDERNUMBER', 'nunique'),
    Avg_Sales=('SALES', 'mean'),
    Total_Revenue=('SALES', 'sum')
).reset_index()

customer_behavior['Is_Outlier_Customer'] = customer_behavior['CUSTOMERNAME'].isin(outlier_customers)

comparison = customer_behavior.groupby('Is_Outlier_Customer').agg(
    Total_Orders=('Total_Orders', 'mean'),
    Avg_Sales=('Avg_Sales', 'mean'),
    Total_Revenue=('Total_Revenue', 'mean')
).rename(index={True: 'Outlier Customer', False: 'Normal Customer'})

print("--- Outlier Customer vs Normal Customer ---")
display(comparison)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
data = comparison.reset_index()
colors = ['#ced4da', '#005088']
metrics = [('Total_Orders', '평균 주문 횟수'), ('Avg_Sales', '평균 단건 매출 ($)'), ('Total_Revenue', '평균 총 매출 기여 ($)')]

for ax, (col, title) in zip(axes, metrics):
    sns.barplot(x='Is_Outlier_Customer', y=col, data=data, ax=ax, palette=colors)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    for p in ax.patches:
        ax.annotate(f'{p.get_height():,.0f}', (p.get_x() + p.get_width()/2, p.get_height()),
                    ha='center', va='bottom', xytext=(0, 5), textcoords='offset points', fontsize=11)

plt.tight_layout()
plt.savefig('../figures/fig10_customer_comparison.png', bbox_inches='tight')
plt.show()

## 8. 파생변수 생성 및 분석

### 8.1 Customer_Priority & PRICE_RATIO 생성

In [ ]:
# Customer_Priority: 고객별 총 매출 > 전체 평균이면 High-Priority
rev_mean = customer_behavior['Total_Revenue'].mean()
customer_behavior['Customer_Priority'] = customer_behavior['Total_Revenue'].apply(
    lambda x: 'High-Priority' if x > rev_mean else 'Standard'
)
df_cleaned = pd.merge(df_cleaned, customer_behavior[['CUSTOMERNAME', 'Customer_Priority']],
                      on='CUSTOMERNAME', how='left')

# PRICE_RATIO: 실거래가 / MSRP (1.0 초과 = 정가 이상, 미만 = 할인)
df_cleaned['UNIT_PRICE'] = df_cleaned['SALES'] / df_cleaned['QUANTITYORDERED']
df_cleaned['PRICE_RATIO'] = df_cleaned['PRICEEACH'] / df_cleaned['MSRP']

# Is_Outlier 플래그
df_cleaned['Is_Outlier'] = df_cleaned.index.isin(all_outliers.index)

print("--- 파생변수 생성 완료 ---")
display(df_cleaned[['CUSTOMERNAME', 'Customer_Priority', 'UNIT_PRICE', 'PRICE_RATIO', 'Is_Outlier']].head())

### 8.2 분석: 이상치는 할인 때문인가? (PRICE_RATIO)

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=df_cleaned, x='Is_Outlier', y='PRICE_RATIO',
            hue='Is_Outlier', palette={False: '#ced4da', True: '#378ADD'}, legend=False)
plt.title('Price Ratio: Normal vs Outlier', fontsize=14, fontweight='bold')
plt.xticks([0, 1], ['Normal', 'Outlier'])
plt.axhline(1.0, color='gray', linestyle='--', alpha=0.6, label='MSRP 기준(1.0)')
plt.legend()
plt.tight_layout()
plt.savefig('../figures/fig11_price_ratio_boxplot.png', bbox_inches='tight')
plt.show()

### 8.3 분석: High-Priority 고객의 이상치 패턴

In [ ]:
df_outliers = df_cleaned[df_cleaned['Is_Outlier'] == True]

plt.figure(figsize=(12, 6))
sns.scatterplot(data=df_outliers, x='QUANTITYORDERED', y='SALES',
                hue='Customer_Priority', style='STATUS', s=100)
plt.title('Outlier Analysis: Quantity vs Sales by Customer Priority', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/fig14_scatter_outlier_priority.png', bbox_inches='tight')
plt.show()

### 8.4 BULK_ORDER_RATIO, REVENUE_CONTRIBUTION, PRICE_CATEGORY 생성

In [ ]:
# BULK_ORDER_RATIO: 제품군별 평균 주문량 대비 비율
avg_qty = df_cleaned.groupby('PRODUCTLINE')['QUANTITYORDERED'].transform('mean')
df_cleaned['BULK_ORDER_RATIO'] = df_cleaned['QUANTITYORDERED'] / avg_qty

# REVENUE_CONTRIBUTION: 고객별 총 매출 / 전체 매출 × 100
total_rev = df_cleaned['SALES'].sum()
customer_rev = df_cleaned.groupby('CUSTOMERNAME')['SALES'].transform('sum')
df_cleaned['REVENUE_CONTRIBUTION'] = (customer_rev / total_rev) * 100

# PRICE_CATEGORY: PRICE_RATIO 기준 3구간
def categorize_price(r):
    if r > 1.05: return 'Premium'
    elif r < 0.95: return 'Discounted'
    return 'Standard'

df_cleaned['PRICE_CATEGORY'] = df_cleaned['PRICE_RATIO'].apply(categorize_price)

print("--- 파생변수 추가 완료 ---")
display(df_cleaned[['CUSTOMERNAME', 'BULK_ORDER_RATIO', 'REVENUE_CONTRIBUTION', 'PRICE_CATEGORY']].head())

### 8.5 분석: 이상치는 대량 주문 때문인가? (BULK_ORDER_RATIO)

In [ ]:
plot_data = df_cleaned.copy()
plot_data['거래 유형'] = plot_data['Is_Outlier'].map({True: 'Outlier', False: 'Normal'})

plt.figure(figsize=(10, 6))
sns.violinplot(data=plot_data, x='거래 유형', y='BULK_ORDER_RATIO',
               hue='거래 유형', palette={'Normal':'#ced4da', 'Outlier':'#ef8a62'},
               inner='quartile', legend=False)
plt.title('Bulk Order Ratio: Normal vs Outlier', fontsize=14, fontweight='bold')
plt.axhline(1.0, color='black', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('../figures/fig12_bulk_order_violin.png', bbox_inches='tight')
plt.show()

### 8.6 분석: VIP일수록 할인받나? (REVENUE_CONTRIBUTION vs PRICE_CATEGORY)

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df_cleaned, x='PRICE_CATEGORY', y='REVENUE_CONTRIBUTION',
            order=['Discounted', 'Standard', 'Premium'], color='#378ADD')
plt.title('Revenue Contribution by Price Category', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/fig13_revenue_by_price_category.png', bbox_inches='tight')
plt.show()

### 8.7 다중공선성 재검증 — 파생변수 포함

In [ ]:
vif_cols2 = ['SALES', 'QUANTITYORDERED', 'PRICEEACH', 'MSRP', 'BULK_ORDER_RATIO', 'PRICE_RATIO']
vif_data2 = df_cleaned[vif_cols2].dropna()
vif_df2 = pd.DataFrame({
    'Feature': vif_data2.columns,
    'VIF': [variance_inflation_factor(vif_data2.values, i) for i in range(len(vif_data2.columns))]
})
print("--- VIF (파생변수 포함) ---")
display(vif_df2.sort_values('VIF', ascending=False))

## 9. 모델링용 데이터셋 준비

### 9.1 인코딩 및 피처 선택

In [ ]:
df_model = df_cleaned.copy()

df_model['Customer_Priority_Code'] = df_model['Customer_Priority'].map({'High-Priority': 1, 'Standard': 0})
df_model['Price_Category_Code'] = df_model['PRICE_CATEGORY'].map({'Discounted': 0, 'Standard': 1, 'Premium': 2})
df_model = pd.get_dummies(df_model, columns=['PRODUCTLINE'], prefix='PL')

final_features = [
    'SALES', 'QUANTITYORDERED', 'PRICE_RATIO',
    'BULK_ORDER_RATIO', 'REVENUE_CONTRIBUTION',
    'Customer_Priority_Code', 'Price_Category_Code'
] + [col for col in df_model.columns if col.startswith('PL_')]

df_final = df_model[final_features + ['Is_Outlier']]

print(f"최종 피처 수: {len(final_features)}")
print(f"데이터셋 shape: {df_final.shape}")
print(f"이상치 비율: {df_final['Is_Outlier'].mean()*100:.2f}%")
display(df_final.head())

### 9.2 CSV 저장

In [ ]:
df_final.to_csv('../data/df_final.csv', index=False)
print("df_final.csv 저장 완료 →  ../data/df_final.csv")